# 📊 TF-IDF — Practical Implementation

## 📌 What is TF-IDF?

**TF-IDF** stands for:

**Term Frequency – Inverse Document Frequency**

It is a numerical text representation technique that measures how important a word is to a particular document relative to the entire collection of documents.

Unlike a basic **Bag of Words (BoW)** representation, which mainly considers word occurrence or frequency, TF-IDF assigns a **weight** to each word.

A word receives:

- A **higher TF-IDF score** when it is important in a particular document but relatively uncommon across the corpus.
- A **lower TF-IDF score** when it appears frequently across many documents.

---

# 🎯 Objective

In this notebook, we will:

1. Load the SMS Spam Collection dataset.
2. Clean and preprocess the SMS messages.
3. Apply **POS-aware WordNet lemmatization**.
4. Convert the processed text into TF-IDF vectors.
5. Understand the TF-IDF feature matrix.
6. Examine the learned vocabulary.
7. Explore TF-IDF with **N-grams**.
8. Compare TF-IDF with Bag of Words.

---

# 🔄 Complete NLP Pipeline

**Raw SMS Messages**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Word Tokenization**

⬇️

**POS Tagging**

⬇️

**Penn Treebank POS → WordNet POS**

⬇️

**POS-Aware Lemmatization**

⬇️

**TF-IDF Vectorization**

⬇️

**Numerical Feature Matrix**

⬇️

**Machine Learning Model**

### In Short

`Raw Text → Preprocessing → POS-Aware Lemmatization → TF-IDF → Numerical Features`

In [1]:
import re
import nltk
import pandas as pd
import numpy as np

from nltk.corpus import stopwords, wordnet
from nltk.tokenize import word_tokenize
from nltk.tag import pos_tag
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
# Download required NLTK resources
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/milind/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/milind/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/milind/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/milind/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

# 📂 1. Load the Dataset

The **SMS Spam Collection** dataset contains SMS messages classified into two categories:

| Label | Meaning |
|---|---|
| `ham` | Normal or legitimate SMS |
| `spam` | Unwanted or promotional SMS |

The dataset contains two main columns:

- `label` — Target category
- `message` — SMS text

The text contained in the `message` column will be preprocessed and converted into numerical TF-IDF features.

In [4]:
# Load the SMS Spam Collection dataset
messages = pd.read_csv(
    "SMSSpamCollection.txt",
    sep="\t",
    names=["label", "message"]
)

messages.head()

,label,message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [5]:
print("Dataset Shape:", messages.shape)
print("\nMissing Values:")
print(messages.isnull().sum())

print("\nClass Distribution:")
print(messages["label"].value_counts())

Dataset Shape: (5572, 2)

Missing Values:
label      0
message    0
dtype: int64

Class Distribution:
label
ham     4825
spam     747
Name: count, dtype: int64


# 🧹 2. Text Cleaning and POS-Aware Lemmatization

Before applying TF-IDF, the raw SMS messages are cleaned and normalized.

The preprocessing pipeline is:

**Raw Message**

⬇️

**Remove Unwanted Characters**

⬇️

**Convert to Lowercase**

⬇️

**Word Tokenization**

⬇️

**POS Tagging**

⬇️

**Convert NLTK POS → WordNet POS**

⬇️

**POS-Aware Lemmatization**

⬇️

**Remove Stopwords**

⬇️

**Processed Message**

## Why POS-Aware Lemmatization?

The correct base form of a word often depends on its grammatical role.

For example:

| Original | POS | Lemma |
|---|---|---|
| `children` | Noun | `child` |
| `running` | Verb | `run` |
| `went` | Verb | `go` |
| `better` | Adjective | `good` |

Basic lemmatization may not correctly reduce verbs and adjectives because `WordNetLemmatizer` uses **noun** as its default POS.

POS-aware lemmatization provides the correct grammatical information before performing lemmatization.

In [6]:
# Store English stopwords in a set for efficient lookup
english_stopwords = set(stopwords.words("english"))

# Initialize WordNet Lemmatizer
lemmatizer = WordNetLemmatizer()

## 🔄 Converting NLTK POS Tags to WordNet POS Tags

NLTK's `pos_tag()` returns **Penn Treebank POS tags**, while `WordNetLemmatizer` expects WordNet POS categories.

The following mapping is used:

| NLTK Tag Starts With | Word Type | WordNet POS |
|---|---|---|
| `J` | Adjective | `wordnet.ADJ` |
| `V` | Verb | `wordnet.VERB` |
| `N` | Noun | `wordnet.NOUN` |
| `R` | Adverb | `wordnet.ADV` |

For example:

`VBG → Verb → wordnet.VERB`

`NNS → Noun → wordnet.NOUN`

`JJR → Adjective → wordnet.ADJ`

`RB → Adverb → wordnet.ADV`

In [7]:
def get_wordnet_pos(tag):
    """
    Convert a Penn Treebank POS tag into a WordNet POS tag.

    Parameters
    ----------
    tag : str
        POS tag generated by NLTK's pos_tag().

    Returns
    -------
    str or None
        Corresponding WordNet POS tag.
        Returns None for unsupported POS categories.
    """

    if tag.startswith("J"):
        return wordnet.ADJ

    elif tag.startswith("V"):
        return wordnet.VERB

    elif tag.startswith("N"):
        return wordnet.NOUN

    elif tag.startswith("R"):
        return wordnet.ADV

    return None

In [8]:
def preprocess_text(text):
    """
    Clean and preprocess text using POS-aware lemmatization.

    Parameters
    ----------
    text : str
        Raw input text.

    Returns
    -------
    str
        Cleaned and POS-aware lemmatized text.
    """

    # Step 1: Remove non-alphabetic characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Step 2: Convert text to lowercase
    text = text.lower()

    # Step 3: Tokenize the text
    words = word_tokenize(text)

    # Step 4: Assign POS tags before removing stopwords
    # This preserves grammatical context for more accurate POS tagging.
    tagged_words = pos_tag(words)

    # Store processed words
    processed_words = []

    for word, tag in tagged_words:

        # Skip English stopwords
        if word in english_stopwords:
            continue

        # Convert NLTK POS to WordNet POS
        wordnet_pos = get_wordnet_pos(tag)

        if wordnet_pos:
            # POS-aware lemmatization
            lemma = lemmatizer.lemmatize(
                word,
                pos=wordnet_pos
            )
        else:
            # Keep unsupported POS unchanged
            lemma = word

        processed_words.append(lemma)

    # Reconstruct the processed text
    return " ".join(processed_words)

# 📚 3. Create the Preprocessed Corpus

The preprocessing function is applied independently to every SMS message.

The resulting collection of processed documents is called the **corpus**.

Each message in the corpus contains cleaned and normalized words that can be passed to `TfidfVectorizer`.

In [9]:
# Apply preprocessing to every SMS message
corpus = [
    preprocess_text(message)
    for message in messages["message"]
]

print("Total Documents:", len(corpus))

Total Documents: 5572


In [10]:
index = 0

print("ORIGINAL MESSAGE")
print("-" * 70)
print(messages["message"].iloc[index])

print("\nPREPROCESSED MESSAGE")
print("-" * 70)
print(corpus[index])

ORIGINAL MESSAGE
----------------------------------------------------------------------
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...

PREPROCESSED MESSAGE
----------------------------------------------------------------------
go jurong point crazy available bugis n great world la e buffet cine get amore wat


# 📐 4. How TF-IDF Works

TF-IDF combines two measurements:

## 1️⃣ Term Frequency — TF

Term Frequency measures how frequently a word appears in a document.

Conceptually:

`TF = Number of times a term appears in a document / Total terms in the document`

A word that appears frequently in a document receives a higher Term Frequency.

---

## 2️⃣ Inverse Document Frequency — IDF

Inverse Document Frequency measures how rare or informative a word is across the complete corpus.

Conceptually:

`IDF = log(Total number of documents / Number of documents containing the term)`

Words appearing in almost every document receive lower importance.

Rare words receive higher importance.

---

## 3️⃣ TF-IDF Score

The final TF-IDF score combines both measurements:

`TF-IDF = TF × IDF`

Therefore:

| Word Behavior | TF-IDF Weight |
|---|---|
| Frequent in one document but rare across corpus | High |
| Frequent across almost every document | Low |
| Does not appear in document | `0` |

### Main Idea

> TF-IDF gives more importance to words that help distinguish one document from other documents.

# 🔢 5. Creating the TF-IDF Feature Matrix

`TfidfVectorizer` performs two main operations:

1. Learns the vocabulary from the corpus.
2. Converts every document into a numerical TF-IDF vector.

Each:

- **Row** represents one SMS message.
- **Column** represents one vocabulary feature.
- **Value** represents the TF-IDF weight of that feature in the document.

The `max_features` parameter limits the vocabulary size.

In [11]:
# Initialize the TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=100
)

# Learn the vocabulary and transform the corpus
X_tfidf = tfidf_vectorizer.fit_transform(corpus)

print("TF-IDF Matrix Shape:", X_tfidf.shape)

TF-IDF Matrix Shape: (5572, 100)


In [12]:
# Display TF-IDF values for the first 5 documents
X_tfidf[:5].toarray()

array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.37427459, 0.        , 0.40385687, 0.        , 0.58578856,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.  

# 📊 6. Understanding the TF-IDF Matrix

Suppose the matrix shape is:

`(5572, 100)`

This means:

- `5572` rows represent **5572 SMS messages**.
- `100` columns represent **100 selected vocabulary features**.

Unlike Count-Based Bag of Words, the matrix contains decimal TF-IDF weights rather than simple word counts.

For example:

| Document | free | prize | message |
|---|---:|---:|---:|
| Document 1 | `0.82` | `0.57` | `0.00` |
| Document 2 | `0.00` | `0.00` | `0.91` |

A higher value indicates that the word has greater importance in that particular document.

> `TfidfVectorizer` returns a sparse matrix because most documents contain only a small portion of the complete vocabulary.

In [13]:
# Create a readable TF-IDF DataFrame for the first 5 messages
tfidf_sample = pd.DataFrame(
    X_tfidf[:5].toarray(),
    columns=tfidf_vectorizer.get_feature_names_out()
)

tfidf_sample

,already,amp,ask,babe,back,buy,call,care,cash,claim,...,wat,way,week,well,win,work,www,yeah,year,yes
0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.594702,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
1,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
2,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.568396,0.0,0.0,0.0,0.0,0.0
3,0.512606,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0
4,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0


# 📚 7. Understanding the TF-IDF Vocabulary

During `fit_transform()`, `TfidfVectorizer` learns a vocabulary from the corpus.

Each selected word is assigned to one column in the feature matrix.

`get_feature_names_out()` returns the features in their actual column order.

The vocabulary may differ depending on preprocessing, vocabulary size, and n-gram configuration.

In [14]:
# Display vocabulary features
tfidf_vectorizer.get_feature_names_out()

array(['already', 'amp', 'ask', 'babe', 'back', 'buy', 'call', 'care',
       'cash', 'claim', 'co', 'come', 'da', 'day', 'dear', 'dont', 'feel',
       'find', 'free', 'friend', 'get', 'give', 'go', 'good', 'great',
       'gt', 'happy', 'hey', 'hi', 'home', 'hope', 'im', 'keep', 'know',
       'last', 'late', 'later', 'leave', 'let', 'life', 'like', 'lor',
       'love', 'lt', 'make', 'meet', 'message', 'min', 'miss', 'mobile',
       'morning', 'msg', 'much', 'na', 'need', 'new', 'night', 'number',
       'oh', 'ok', 'one', 'phone', 'pick', 'please', 'pls', 'prize',
       'really', 'reply', 'right', 'say', 'see', 'send', 'sorry', 'still',
       'stop', 'take', 'tell', 'text', 'thing', 'think', 'time', 'today',
       'tomorrow', 'tone', 'try', 'txt', 'ur', 'wait', 'wan', 'want',
       'wat', 'way', 'week', 'well', 'win', 'work', 'www', 'yeah', 'year',
       'yes'], dtype=object)

In [15]:
for index, feature in enumerate(
    tfidf_vectorizer.get_feature_names_out()
):
    print(f"{index:<5} ---> {feature}")

0     ---> already
1     ---> amp
2     ---> ask
3     ---> babe
4     ---> back
5     ---> buy
6     ---> call
7     ---> care
8     ---> cash
9     ---> claim
10    ---> co
11    ---> come
12    ---> da
13    ---> day
14    ---> dear
15    ---> dont
16    ---> feel
17    ---> find
18    ---> free
19    ---> friend
20    ---> get
21    ---> give
22    ---> go
23    ---> good
24    ---> great
25    ---> gt
26    ---> happy
27    ---> hey
28    ---> hi
29    ---> home
30    ---> hope
31    ---> im
32    ---> keep
33    ---> know
34    ---> last
35    ---> late
36    ---> later
37    ---> leave
38    ---> let
39    ---> life
40    ---> like
41    ---> lor
42    ---> love
43    ---> lt
44    ---> make
45    ---> meet
46    ---> message
47    ---> min
48    ---> miss
49    ---> mobile
50    ---> morning
51    ---> msg
52    ---> much
53    ---> na
54    ---> need
55    ---> new
56    ---> night
57    ---> number
58    ---> oh
59    ---> ok
60    ---> one
61    ---> phone
62    ---> pick
63

# 🔗 8. TF-IDF with N-Grams

TF-IDF can also assign importance scores to sequences of words.

## Unigrams

Individual words:

`machine`

`learning`

## Bigrams

Two consecutive words:

`machine learning`

## Trigrams

Three consecutive words:

`machine learning model`

The `ngram_range` parameter determines which combinations are included.

| `ngram_range` | Features |
|---|---|
| `(1, 1)` | Unigrams only |
| `(2, 2)` | Bigrams only |
| `(3, 3)` | Trigrams only |
| `(1, 2)` | Unigrams + Bigrams |
| `(1, 3)` | Unigrams + Bigrams + Trigrams |
| `(2, 3)` | Bigrams + Trigrams |

N-grams can capture short phrases and local context that individual words may miss.

However, increasing the n-gram range also increases the number of possible features.

In [16]:
# Create a TF-IDF vectorizer using only bigrams
bigram_tfidf_vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(2, 2)
)

# Create bigram TF-IDF features
X_bigram_tfidf = bigram_tfidf_vectorizer.fit_transform(
    corpus
)

print(
    "Bigram TF-IDF Matrix Shape:",
    X_bigram_tfidf.shape
)

Bigram TF-IDF Matrix Shape: (5572, 100)


In [17]:
bigram_tfidf_vectorizer.get_feature_names_out()

array(['account statement', 'attempt contact', 'await collection',
       'call claim', 'call customer', 'call identifier', 'call land',
       'call landline', 'call later', 'call mobileupd', 'call optout',
       'call per', 'camera phone', 'cash prize', 'chance win',
       'claim call', 'claim ur', 'claim valid', 'co uk', 'come back',
       'come home', 'customer service', 'date service', 'decimal gt',
       'dont know', 'double min', 'draw show', 'every week', 'first time',
       'free call', 'free entry', 'free text', 'get back', 'gift voucher',
       'go get', 'go home', 'go sleep', 'gon na', 'good afternoon',
       'good morning', 'good night', 'great day', 'gt lt', 'gt min',
       'guarantee call', 'gud mrng', 'gud ni', 'happy new', 'hi hi',
       'holiday cash', 'hope good', 'identifier code', 'land line',
       'land row', 'last night', 'let know', 'like lt', 'line claim',
       'lt decimal', 'lt gt', 'miss call', 'mobile number',
       'national rate', 'nd attempt

In [18]:
X_bigram_tfidf[:5].toarray()

array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.

In [19]:
# Create TF-IDF features using both unigrams and bigrams
unigram_bigram_vectorizer = TfidfVectorizer(
    max_features=500,
    ngram_range=(1, 2)
)

X_unigram_bigram = unigram_bigram_vectorizer.fit_transform(
    corpus
)

print(
    "Unigram + Bigram Matrix Shape:",
    X_unigram_bigram.shape
)

Unigram + Bigram Matrix Shape: (5572, 500)


In [20]:
unigram_bigram_vectorizer.get_feature_names_out()[:50]

array(['able', 'abt', 'account', 'actually', 'address', 'afternoon',
       'age', 'ah', 'aight', 'already', 'alright', 'also', 'always',
       'amp', 'another', 'answer', 'anything', 'anyway', 'apply', 'ard',
       'around', 'ask', 'attempt', 'await', 'await collection', 'award',
       'away', 'babe', 'baby', 'back', 'bad', 'beautiful', 'bed',
       'believe', 'best', 'big', 'birthday', 'bit', 'bonus', 'book',
       'bore', 'box', 'boy', 'break', 'bring', 'brother', 'bt', 'bus',
       'busy', 'buy'], dtype=object)

# ⚖️ 9. Bag of Words vs TF-IDF

Both Bag of Words and TF-IDF convert text into numerical vectors, but they assign values differently.

| Feature | Bag of Words | TF-IDF |
|---|---|---|
| Representation | Word count or presence | Weighted importance |
| Common words | Can receive high counts | Usually receive lower weights |
| Rare informative words | No special importance | Can receive higher importance |
| Output values | Integer counts or binary values | Decimal weights |
| Vocabulary | Yes | Yes |
| Word order | Not preserved by default | Not preserved by default |
| N-gram support | Yes | Yes |

## Example

Suppose the word `message` appears in almost every document.

### Bag of Words

A document containing `message` five times may assign:

`message → 5`

### TF-IDF

Because `message` occurs across many documents, its IDF component may be low.

Therefore:

`message → Low TF-IDF Weight`

Now consider a less common but highly relevant word such as:

`lottery`

If it appears frequently in one spam message but rarely across the corpus:

`lottery → High TF-IDF Weight`

### Main Difference

**Bag of Words asks:**

> How many times does this word appear?

**TF-IDF asks:**

> How important is this word to this document compared with the entire corpus?

# ⚠️ Important Practical Considerations

## 1. Keep the Matrix Sparse

Avoid converting the complete TF-IDF matrix using:

`X = vectorizer.fit_transform(corpus).toarray()`

for large datasets.

Instead, use:

`X = vectorizer.fit_transform(corpus)`

and convert only small samples when necessary.

---

## 2. Fit Only on Training Data

In a real machine learning project, `TfidfVectorizer` should be fitted only on the training data.

Correct workflow:

**Training Text**

⬇️

`fit_transform()`

**Testing Text**

⬇️

`transform()`

The test data should never be used to learn the vocabulary or IDF values.

This prevents **data leakage**.

---

## 3. Preprocessing Should Be Validated Experimentally

POS-aware lemmatization is linguistically more accurate, but more preprocessing does not always guarantee better classification performance.

For a real NLP model, compare approaches such as:

- Raw text + TF-IDF
- Stopword removal + TF-IDF
- Snowball stemming + TF-IDF
- Basic lemmatization + TF-IDF
- POS-aware lemmatization + TF-IDF

The final preprocessing strategy should be selected using model evaluation results.

# 🧠 Key Takeaways

- **TF-IDF** stands for Term Frequency–Inverse Document Frequency.
- TF measures how frequently a word appears in a document.
- IDF measures how rare or informative a word is across the corpus.
- TF-IDF assigns higher importance to words that help distinguish documents.
- `TfidfVectorizer` converts processed text into numerical feature vectors.
- POS-aware lemmatization can improve linguistic normalization before vectorization.
- `max_features` controls the maximum number of selected vocabulary features.
- `ngram_range` allows TF-IDF to represent individual words and multi-word sequences.
- `(1, 1)` represents unigrams.
- `(2, 2)` represents bigrams.
- `(1, 2)` represents unigrams and bigrams.
- Sparse matrices should generally be preserved for memory efficiency.
- In machine learning projects, the vectorizer must be fitted only on training data to prevent data leakage.
- The best preprocessing and vectorization configuration should be selected through model evaluation.

---

# 🏆 Complete TF-IDF Pipeline

**Raw SMS Messages**

⬇️

**Text Cleaning**

⬇️

**Lowercasing**

⬇️

**Tokenization**

⬇️

**POS Tagging**

⬇️

**POS Mapping**

⬇️

**POS-Aware Lemmatization**

⬇️

**TF-IDF Vectorization**

⬇️

**Numerical Feature Matrix**

⬇️

**Machine Learning Model**

## 📌 In Short

`Raw Text → Clean → Tokenize → POS Tag → Lemmatize → TF-IDF → Model`